# Data Coverage

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option

In [ ]:
pd.options.mode.copy_on_write = True

Allow reloading of custom Python classes

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

### Calculating Coverage

In [ ]:
# Calculate active weeks
active_weeks_dict = {}
for loc_id, df in sales_and_menu_data.items():
    df = df.copy()
    df.index = df.index.tz_localize(None) # Remove timezone info
    df['Week'] = df.index.to_period('W')
    active_weeks_dict[loc_id] = set(df.groupby('Week').size().index.tolist())

### Coverage Visual

In [ ]:
# Visualizing with gaps for inactive weeks
plt.figure(figsize=(14, 8))

# Loop through every active week
for base_name, active_weeks in active_weeks_dict.items(): # active_weeks is all the active weeks in a single restaurant

    # Change to restaurant ID
    loc_id = re.sub(r'_sales_and_menu$', '', base_name)

    # For every active week
    for week in active_weeks:

        # Place a blue dot
        plt.hlines(y=loc_id, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

    # Index into the promotional items for this restaurant
    promo_datetime = before_after_details.loc[loc_id,'cross_over_date']

    # Place a red circle for the promotional item
    plt.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
plt.title('Weekly Activity for Each Restaurant with Gaps for Inactive Weeks')
plt.xlabel('Date')
plt.ylabel('Restaurant ID')
plt.yticks()
plt.tight_layout()
plt.show()

### Data Density

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
data_density_list = []
for loc_id, df in list(sales_and_menu_data.items()):
    
    # Copy to prevent removal of timezone
    df = df.copy()
    df.index = df.index.tz_localize(None)

    # Identify introductin date
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date']

    # Resample the time series
    half_day_counts = df['item_name'].resample('12H').count()
    daily_counts = df['item_name'].resample('D').count()
    weekly_counts = df['item_name'].resample('W-MON').count()
    monthly_counts = df['item_name'].resample('M').count()

    # Half day stats
    half_day_coverage_ratio = round(100 * (half_day_counts > 1).sum() / half_day_counts.size)/100
    half_day_mean = round(np.mean(half_day_counts))
    half_day_sd = round(np.std(half_day_counts)) 

    # Day stats
    daily_coverage_ratio = round(100 * (daily_counts > 1).sum() / daily_counts.size)/100
    daily_mean = round(np.mean(daily_counts))
    daily_sd = round(np.std(daily_counts))

    # Week stats
    weekly_coverage_ratio = round(100 * (weekly_counts > 1).sum() / weekly_counts.size)/100
    weekly_mean = round(np.mean(weekly_counts))
    weekly_sd = round(np.std(weekly_counts))

    # Month stats
    monthly_coverage_ratio = round(100 * (monthly_counts > 1).sum() / monthly_counts.size)/100
    monthly_mean = round(np.mean(monthly_counts))
    monthly_sd = round(np.std(monthly_counts))

    # Aggregate for summary
    row = {'loc_id': loc_id,
           'hd_coverage': half_day_coverage_ratio,
           'd_coverage': daily_coverage_ratio,
           'w_coverage': weekly_coverage_ratio,
           'm_coverage': monthly_coverage_ratio,
           'hd_mean': half_day_mean, 
           'hd_sd': half_day_sd, 
           'd_mean': daily_mean, 
           'd_sd': daily_sd, 
           'w_mean': weekly_mean, 
           'w_sd': weekly_sd,
           'm_mean': monthly_mean, 
           'm_sd': monthly_sd, }
    
    data_density_list.append(row)

data_density = pd.DataFrame(data_density_list)

View

In [ ]:
data_density.sort_values('hd_coverage')

### Buffer Data Before and After Promo

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
buffer_data_list = []
for loc_id, df in sales_and_menu_data.items():
    
    # Copy to prevent removal of timezone
    df = df.copy()
    df.index = df.index.tz_localize(None)

    # Identify introductin date
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date']

    # Create an offset because we want data two months before and after the promotional introduction date
    before_limit = promo_datetime - pd.DateOffset(months=2)
    after_limit = promo_datetime + pd.DateOffset(months=2)

    # Slice to before and after
    before_data = df.loc[:promo_datetime]
    after_data = df.loc[promo_datetime:]

    # Bound by two months
    two_months_prior = before_data.loc[before_limit:]
    two_months_after = after_data.loc[:after_limit]

    # Number of data entries before and after
    before_data_count = before_data.shape[0]
    after_data_count = after_data.shape[0]

    # Number of active weeks before and after
    before_week_count = (0 < before_data['item_name'].resample('W-MON').count()).sum()
    after_week_count = (0 < after_data['item_name'].resample('W-MON').count()).sum()

    # Number of data entries within the bounds
    two_months_prior_data_count = two_months_prior.shape[0]
    two_months_after_data_count = two_months_after.shape[0]

    # Number of active weeks within the bounds
    two_months_prior_week_count = (0 < two_months_prior['item_name'].resample('W-MON').count()).sum()
    two_months_after_week_count = (0 < two_months_after['item_name'].resample('W-MON').count()).sum()

    # Data coverage within the bounds
    two_months_prior_half_day_counts = before_data['item_name'].resample('12H').count()
    two_months_prior_half_day_coverage_ratio = round(100 * (two_months_prior_half_day_counts > 1).sum() / two_months_prior_half_day_counts.size)/100
    two_months_after_half_day_counts = after_data['item_name'].resample('12H').count()
    two_months_after_half_day_coverage_ratio = round(100 * (two_months_after_half_day_counts > 1).sum() / two_months_after_half_day_counts.size)/100

    # Aggregate for summary
    row = {'loc_id': loc_id, 
           'b_entries': before_data_count, 
           'b_weeks': before_week_count, 
           'b2m_entries': two_months_prior_data_count,
           'b2m_weeks': two_months_prior_week_count,
           'b2m_hd_coverage': two_months_prior_half_day_coverage_ratio,
           'a_entries': after_data_count, 
           'a_weeks': after_week_count, 
           'a2m_entries': two_months_after_data_count,
           'a2m_weeks': two_months_after_week_count,
           'a2m_hd_coverage': two_months_after_half_day_coverage_ratio}
    buffer_data_list.append(row)
    
buffer_data = pd.DataFrame(buffer_data_list)

View

In [ ]:
buffer_data.sort_values('b2m_hd_coverage')

In [ ]:
weekly_pb_list = []
monthly_pb_list = []
yearly_pb_list = []

visual_dict = {}

for loc_id, df in sales_and_menu_data.items():

    fig, ax = plt.subplots()  # Creates a single subplot


    # df['week'] = df.index.isocalendar().week
    # df['month'] = df.index.month
    # df['year'] = df.index.year
    plant_based = fdf(df).filter('is_plant_based','Yes')
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id]['cross_over_date'])
    # weekly_pb_list.append(plant_based.groupby('week')['item_name'].count())
    # monthly_pb_list.append(plant_based.groupby('month')['item_name'].count())
    # yearly_pb_list.append(plant_based.groupby('year')['item_name'].count())

    ax.plot(plant_based.resample('W')['item_name'].count() / df.resample('W')['item_name'].count())
    ax.axvline(x=promo_datetime, color='red', linestyle='--')
    ax.set_title(loc_id)
    visual_dict[loc_id] = ax
